# Notebook 3 — Filter Anomalies & Export Excel Report

Takes the raw scan results from Notebook 2 and:
- Applies quality filters to find only statistically meaningful anomalies
- Ranks and deduplicates results
- Exports a clean Excel report with 3 sheets:
  - **Sheet 1:** All anomalies ranked by Sharpe
  - **Sheet 2:** Best anomaly per sector
  - **Sheet 3:** Monthly heatmap (sector × month avg return)

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
from openpyxl import load_workbook
from openpyxl.styles import (
    PatternFill, Font, Alignment, Border, Side
)
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


In [2]:
# ── Configuration — Tune your filters here ────────────────────────────────────
#
# These thresholds decide what counts as a real anomaly vs noise.
# After looking at your distribution output from Notebook 2,
# adjust these to be tighter or looser.
#
# Defaults are conservative — better to miss weak signals than trade noise.

FILTERS = {
    'min_win_rate'     : 65.0,   # % of years the window was profitable
    'min_avg_return'   : 1.5,    # % average return across years
    'min_sharpe'       : 0.8,    # risk-adjusted return (lower = more results)
    'max_drawdown'     : -5.0,   # worst intra-window drop allowed (e.g. -5%)
    'min_observations' : 4,      # minimum years of data for this window
}

INPUT_PATH  = os.path.join('data', 'raw_scan_results.csv')
OUTPUT_DIR  = 'output'
OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'anomalies.xlsx')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Filter thresholds:')
for k, v in FILTERS.items():
    print(f'  {k}: {v}')

Filter thresholds:
  min_win_rate: 65.0
  min_avg_return: 1.5
  min_sharpe: 0.8
  max_drawdown: -5.0
  min_observations: 4


In [3]:
# ── Load Raw Results ──────────────────────────────────────────────────────────

print(f'Loading raw results from {INPUT_PATH} ...')
df = pd.read_csv(INPUT_PATH)
print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')

Loading raw results from data\raw_scan_results.csv ...
Loaded 219,000 rows
Columns: ['sector', 'month', 'day', 'start_label', 'window_days', 'end_label', 'avg_return', 'win_rate', 'max_drawdown', 'sharpe', 'n_obs']


In [4]:
# ── Apply Filters ─────────────────────────────────────────────────────────────

filtered = df[
    (df['win_rate']     >= FILTERS['min_win_rate'])     &
    (df['avg_return']   >= FILTERS['min_avg_return'])   &
    (df['sharpe']       >= FILTERS['min_sharpe'])       &
    (df['max_drawdown'] >= FILTERS['max_drawdown'])     &
    (df['n_obs']        >= FILTERS['min_observations'])
].copy()

print(f'After filtering: {len(filtered):,} anomalies')
print(f'Reduction: {len(df):,} → {len(filtered):,} ({len(filtered)/len(df)*100:.2f}% kept)')

if len(filtered) == 0:
    print('\n⚠ No results passed filters. Try relaxing thresholds in the config cell.')
    print('  Suggestions:')
    print('  - Lower min_win_rate to 60')
    print('  - Lower min_avg_return to 1.0')
    print('  - Lower min_sharpe to 0.5')

After filtering: 4,648 anomalies
Reduction: 219,000 → 4,648 (2.12% kept)


In [5]:
# ── Rank & Clean ──────────────────────────────────────────────────────────────
# Add a composite score: weighted combination of the 4 metrics
# This gives a single number to rank anomalies against each other

# Normalize each metric to 0-1 scale within the filtered set
def minmax(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

filtered['score_return']   = minmax(filtered['avg_return'])
filtered['score_winrate']  = minmax(filtered['win_rate'])
filtered['score_sharpe']   = minmax(filtered['sharpe'])
# For drawdown: less negative = better, so invert
filtered['score_drawdown'] = minmax(filtered['max_drawdown'])

# Composite score — weights reflect what matters most for trading
filtered['composite_score'] = (
    filtered['score_sharpe']   * 0.35 +   # risk-adjusted return — most important
    filtered['score_winrate']  * 0.30 +   # consistency
    filtered['score_return']   * 0.25 +   # raw return
    filtered['score_drawdown'] * 0.10     # drawdown control
).round(4)

# Sort by composite score
filtered = filtered.sort_values('composite_score', ascending=False)
filtered['rank'] = range(1, len(filtered) + 1)

# Clean display columns
display_cols = [
    'rank', 'sector', 'start_label', 'window_days', 'end_label',
    'avg_return', 'win_rate', 'max_drawdown', 'sharpe',
    'n_obs', 'composite_score'
]

sheet1 = filtered[display_cols].copy()
sheet1.columns = [
    'Rank', 'Sector', 'Window Start', 'Window Days', 'Window End',
    'Avg Return %', 'Win Rate %', 'Max Drawdown %', 'Sharpe',
    'Years of Data', 'Score'
]

print(f'Sheet 1 preview — Top 10 anomalies:')
print(sheet1.head(10).to_string(index=False))

Sheet 1 preview — Top 10 anomalies:
 Rank       Sector Window Start  Window Days Window End  Avg Return %  Win Rate %  Max Drawdown %  Sharpe  Years of Data  Score
    1   Nifty FMCG       Mar 26           50     May 15        8.0393       100.0         -2.9935  2.0954             10 0.7903
    2   Nifty FMCG       Mar 27           47     May 13        8.4251       100.0         -2.7779  1.9648             10 0.7806
    3   Nifty FMCG       Mar 26           49     May 14        7.8838       100.0         -2.9935  2.0608             10 0.7788
    4   Nifty FMCG       Mar 26           48     May 13        8.0546       100.0         -2.9935  1.9940             10 0.7709
    5   Nifty FMCG       Mar 27           48     May 14        7.7088       100.0         -2.7779  1.9925             10 0.7645
    6   Nifty FMCG       Mar 26           51     May 16        7.7084       100.0         -2.9935  2.0098             10 0.7636
    7   Nifty FMCG       Mar 26           47     May 12        8.431

In [6]:
# ── Sheet 2: Best Anomaly Per Sector ──────────────────────────────────────────
# For each sector, pick the single highest-scoring anomaly

sheet2 = (
    filtered
    .sort_values('composite_score', ascending=False)
    .groupby('sector')
    .first()
    .reset_index()
)[display_cols]

sheet2.columns = [
    'Rank', 'Sector', 'Window Start', 'Window Days', 'Window End',
    'Avg Return %', 'Win Rate %', 'Max Drawdown %', 'Sharpe',
    'Years of Data', 'Score'
]

sheet2 = sheet2.sort_values('Score', ascending=False).reset_index(drop=True)
sheet2['Rank'] = range(1, len(sheet2) + 1)

print(f'Sheet 2 — Best anomaly per sector:')
print(sheet2.to_string(index=False))

Sheet 2 — Best anomaly per sector:
 Rank                   Sector Window Start  Window Days Window End  Avg Return %  Win Rate %  Max Drawdown %  Sharpe  Years of Data  Score
    1               Nifty FMCG       Mar 26           50     May 15        8.0393       100.0         -2.9935  2.0954             10 0.7903
    2             Nifty Pharma       Aug 29           10     Sep 08        2.0120       100.0         -0.7228  2.5867             10 0.7509
    3             Nifty Realty       Mar 29            1     Mar 30        1.8259       100.0          0.0000  2.2866             11 0.7010
    4 Nifty Financial Services       May 16           53     Jul 08        8.6649       100.0         -3.1682  1.4002             10 0.6694
    5                 Nifty IT       Nov 04           50     Dec 24        9.4214       100.0         -4.9320  1.3102             10 0.6392
    6               Nifty Bank       Jun 15           22     Jul 07        3.5476       100.0         -2.7450  1.9505        

In [7]:
# ── Sheet 3: Monthly Heatmap ──────────────────────────────────────────────────
# Average return per sector per calendar month
# Gives a bird's eye view of seasonal patterns

month_names = [
    'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
]

# Use ALL windows (not just filtered) for the heatmap — gives fuller picture
heatmap_data = (
    df.groupby(['sector', 'month'])['avg_return']
    .mean()
    .reset_index()
)

sheet3 = heatmap_data.pivot(index='sector', columns='month', values='avg_return')
sheet3.columns = [month_names[m - 1] for m in sheet3.columns]
sheet3 = sheet3.round(3)
sheet3.index.name = 'Sector'
sheet3 = sheet3.reset_index()

print('Sheet 3 — Monthly return heatmap:')
print(sheet3.to_string(index=False))

Sheet 3 — Monthly return heatmap:
                  Sector    Jan    Feb    Mar   Apr   May   Jun   Jul   Aug    Sep    Oct   Nov    Dec
              Nifty Auto -1.266 -2.332  4.442 6.209 5.300 2.624 2.378 2.665  0.400  1.385 2.201  1.385
              Nifty Bank -0.075 -1.711  2.532 3.370 3.977 2.474 0.899 1.860  2.864  4.447 2.067  0.908
            Nifty Energy  1.190  2.443  4.710 3.420 3.226 3.394 3.169 2.160  1.188  1.752 1.990  1.223
              Nifty FMCG -1.148  0.165  4.395 3.156 3.937 3.808 1.456 0.216 -1.340  0.405 1.114 -0.238
Nifty Financial Services -0.056 -1.097  2.951 3.609 4.394 3.205 1.421 1.783  2.402  4.096 1.915  0.636
                Nifty IT -2.505 -4.371 -1.036 1.753 3.669 4.297 3.879 1.791  0.842  2.812 4.806  1.951
             Nifty Media -3.748 -3.310  0.853 1.813 3.722 1.492 2.261 2.243 -0.758  0.038 0.191 -3.025
             Nifty Metal  1.136  2.692  5.107 3.888 3.798 4.076 4.764 3.288  2.203  4.410 4.201  2.442
            Nifty Pharma -0.914  0.797 

In [8]:
# ── Export to Excel ───────────────────────────────────────────────────────────

print(f'Writing Excel report to {OUTPUT_PATH} ...')

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    sheet1.to_excel(writer, sheet_name='All Anomalies',      index=False)
    sheet2.to_excel(writer, sheet_name='Best Per Sector',    index=False)
    sheet3.to_excel(writer, sheet_name='Monthly Heatmap',    index=False)

print('Raw export done. Applying formatting...')

Writing Excel report to output\anomalies.xlsx ...
Raw export done. Applying formatting...


In [9]:
# ── Apply Excel Formatting ────────────────────────────────────────────────────

wb = load_workbook(OUTPUT_PATH)

# ── Color palette ──────────────────────────────────────────────────────────
COLOR_HEADER     = '1F3864'   # dark navy — header background
COLOR_HEADER_FT  = 'FFFFFF'   # white text
COLOR_ROW_ALT    = 'EEF2FF'   # light blue — alternate rows
COLOR_POSITIVE   = 'C6EFCE'   # light green
COLOR_NEGATIVE   = 'FFC7CE'   # light red
COLOR_BEST       = 'FFD700'   # gold — top ranked rows

header_fill = PatternFill('solid', fgColor=COLOR_HEADER)
alt_fill    = PatternFill('solid', fgColor=COLOR_ROW_ALT)
pos_fill    = PatternFill('solid', fgColor=COLOR_POSITIVE)
neg_fill    = PatternFill('solid', fgColor=COLOR_NEGATIVE)
gold_fill   = PatternFill('solid', fgColor=COLOR_BEST)

header_font = Font(bold=True, color=COLOR_HEADER_FT, size=11)
bold_font   = Font(bold=True)

center = Alignment(horizontal='center', vertical='center')
left   = Alignment(horizontal='left',   vertical='center')

thin_border = Border(
    bottom=Side(style='thin', color='CCCCCC')
)

def format_sheet(ws, df_ref, col_widths=None):
    """Apply consistent formatting to a worksheet."""
    
    # Header row
    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = center
        cell.border    = thin_border
    
    # Data rows
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        fill = alt_fill if row_idx % 2 == 0 else PatternFill()  # alternate shading
        for cell in row:
            cell.alignment = center
            cell.border    = thin_border
            if cell.fill.patternType is None or cell.fill.fgColor.rgb == '00000000':
                cell.fill = fill
    
    # Freeze top row
    ws.freeze_panes = 'A2'
    
    # Row height
    ws.row_dimensions[1].height = 20
    for i in range(2, ws.max_row + 1):
        ws.row_dimensions[i].height = 16
    
    # Auto column width
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = min(max(max_len + 3, 10), 30)


# ── Format Sheet 1: All Anomalies ─────────────────────────────────────────
ws1 = wb['All Anomalies']
format_sheet(ws1, sheet1)

# Highlight top 10 rows in gold
for row in ws1.iter_rows(min_row=2, max_row=min(11, ws1.max_row)):
    for cell in row:
        cell.fill = gold_fill

# Color-code Avg Return % column (col 6) and Max Drawdown % (col 8)
ret_col = 6
dd_col  = 8
for row in ws1.iter_rows(min_row=2, max_row=ws1.max_row):
    ret_cell = row[ret_col - 1]
    dd_cell  = row[dd_col  - 1]
    
    if ret_cell.value is not None:
        ret_cell.fill = pos_fill if float(ret_cell.value) > 0 else neg_fill
    if dd_cell.value is not None:
        dd_cell.fill  = pos_fill if float(dd_cell.value)  > -2 else neg_fill


# ── Format Sheet 2: Best Per Sector ───────────────────────────────────────
ws2 = wb['Best Per Sector']
format_sheet(ws2, sheet2)

# Highlight #1 ranked sector in gold
for cell in ws2[2]:
    cell.fill = gold_fill
    cell.font = bold_font


# ── Format Sheet 3: Monthly Heatmap ───────────────────────────────────────
ws3 = wb['Monthly Heatmap']
format_sheet(ws3, sheet3)

# Apply a green-white-red color scale to the numeric cells (cols 2 onward)
from openpyxl.utils import get_column_letter
last_col = get_column_letter(ws3.max_column)
last_row = ws3.max_row

color_scale = ColorScaleRule(
    start_type='min',  start_color='F8696B',   # red   = lowest return
    mid_type='num',    mid_value=0,             # white = 0%
                       mid_color='FFFFFF',
    end_type='max',    end_color='63BE7B'       # green = highest return
)
ws3.conditional_formatting.add(f'B2:{last_col}{last_row}', color_scale)


# ── Save ──────────────────────────────────────────────────────────────────
wb.save(OUTPUT_PATH)
print(f'\n✅ Excel report saved: {OUTPUT_PATH}')
print(f'   Sheet 1 — All Anomalies  : {len(sheet1):,} rows')
print(f'   Sheet 2 — Best Per Sector: {len(sheet2)} rows')
print(f'   Sheet 3 — Monthly Heatmap: {len(sheet3)} sectors × 12 months')


✅ Excel report saved: output\anomalies.xlsx
   Sheet 1 — All Anomalies  : 4,648 rows
   Sheet 2 — Best Per Sector: 10 rows
   Sheet 3 — Monthly Heatmap: 10 sectors × 12 months


In [10]:
# ── Final Summary ─────────────────────────────────────────────────────────────
# Print a clean readable summary of the best findings

print('═' * 65)
print('         TOP 15 ANOMALIES FOUND')
print('═' * 65)

top15 = sheet1.head(15)
for _, row in top15.iterrows():
    print(
        f"#{int(row['Rank']):>2}  {row['Sector']:<28} "
        f"{row['Window Start']} → {row['Window End']} "
        f"({int(row['Window Days'])}d) | "
        f"Return: {row['Avg Return %']:>5.2f}%  "
        f"WinRate: {row['Win Rate %']:>5.1f}%  "
        f"Sharpe: {row['Sharpe']:>5.2f}"
    )

print('═' * 65)
print(f'\nTotal anomalies found : {len(sheet1):,}')
print(f'Sectors with anomalies: {sheet1["Sector"].nunique()}')
print(f'\nOpen {OUTPUT_PATH} to explore the full report.')

═════════════════════════════════════════════════════════════════
         TOP 15 ANOMALIES FOUND
═════════════════════════════════════════════════════════════════
# 1  Nifty FMCG                   Mar 26 → May 15 (50d) | Return:  8.04%  WinRate: 100.0%  Sharpe:  2.10
# 2  Nifty FMCG                   Mar 27 → May 13 (47d) | Return:  8.43%  WinRate: 100.0%  Sharpe:  1.96
# 3  Nifty FMCG                   Mar 26 → May 14 (49d) | Return:  7.88%  WinRate: 100.0%  Sharpe:  2.06
# 4  Nifty FMCG                   Mar 26 → May 13 (48d) | Return:  8.05%  WinRate: 100.0%  Sharpe:  1.99
# 5  Nifty FMCG                   Mar 27 → May 14 (48d) | Return:  7.71%  WinRate: 100.0%  Sharpe:  1.99
# 6  Nifty FMCG                   Mar 26 → May 16 (51d) | Return:  7.71%  WinRate: 100.0%  Sharpe:  2.01
# 7  Nifty FMCG                   Mar 26 → May 12 (47d) | Return:  8.43%  WinRate: 100.0%  Sharpe:  1.87
# 8  Nifty FMCG                   Mar 25 → May 14 (50d) | Return:  8.76%  WinRate: 100.0%  Sharpe:  1